# 01 — Download HEST-1k slides

GeneRAG needs **spatial-transcriptomics slides** (one or more `.h5ad` per tissue) and the
**corresponding H&E images** (`.tif`). The easiest source is the public
[HEST-1k](https://huggingface.co/datasets/MahmoodLab/hest) dataset.

This notebook downloads the slides used in the GeneRAG paper for one organ at a time. The
Prostate (PRAD) cohort is shown end-to-end as the default example; the other three
(HER2ST / Breast, Kidney, Mouse Brain) are included as commented blocks.

**Prerequisites**
* A HuggingFace account with access to `MahmoodLab/hest` (most slides are public).
* An access token exported as `HF_TOKEN` *or* passed inline below.
* `datasets`, `huggingface_hub`, `pandas`.

In [ ]:
# !pip install datasets huggingface_hub pandas

In [ ]:
import os
from huggingface_hub import login

# Prefer the env var so the notebook stays anonymous; fall back to an explicit value.
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # paste your token here if not exported
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print('No HF_TOKEN found — public slides may still download but private ones will fail.')

In [ ]:
import pandas as pd

# HEST-1k metadata: maps slide ids to tissue, donor, modality, etc.
meta_df = pd.read_csv('hf://datasets/MahmoodLab/hest/HEST_v1_1_0.csv')
print('HEST-1k records:', len(meta_df))
meta_df.head(3)

## Choose an organ

Each cohort below downloads a few-GB subset under `./hest1k_datasets/<organ>/`. Run
*only the cell you need* — the others are intentionally provided as alternate cohorts.

In [ ]:
from datasets import load_dataset

ORGAN = 'PRAD'                                  # change to 'kidney', 'her2st', 'mouse_brain', ...
DATA_ROOT = './hest1k_datasets'
data_path = os.path.join(DATA_ROOT, ORGAN)
os.makedirs(data_path, exist_ok=True)

In [ ]:
# --- Prostate (PRAD) -------------------------------------------------
# Test slide used in the paper: MEND145.
ids_to_query = [f'MEND{i}' for i in range(139, 163)]
patterns = [f'*{sid}[_.]**' for sid in ids_to_query]

load_dataset('MahmoodLab/hest', cache_dir=data_path, patterns=patterns)

### Other cohorts (uncomment to use)

In [ ]:
# --- Kidney ----------------------------------------------------------
# Test slide used in the paper: NCBI697.
# ids = meta_df.loc[meta_df['dataset_title'].str.startswith(
#     'Spatial localization with Spatial Transcriptomics for an atlas of healthy and injured cell states'
# ), 'id'].tolist()
# patterns = [f'*{sid}[_.]**' for sid in ids]
# load_dataset('MahmoodLab/hest', cache_dir='./hest1k_datasets/kidney', patterns=patterns)

In [ ]:
# --- Breast (HER2ST) ------------------------------------------------
# Test slides used in the paper: SPA148 (B1) or SPA123 (G2).
# ids = meta_df.loc[meta_df['dataset_title'].str.startswith(
#     'Spatial deconvolution of'
# ), 'id'].tolist()
# patterns = [f'*{sid}[_.]**' for sid in ids]
# load_dataset('MahmoodLab/hest', cache_dir='./hest1k_datasets/her2st', patterns=patterns)

In [ ]:
# --- Mouse Brain ----------------------------------------------------
# Test slide used in the paper: NCBI667.
# ids = meta_df.loc[
#     meta_df['dataset_title'].str.startswith('Spatial Multimodal Analysis')
#     & (meta_df['disease_state'] == 'Healthy'), 'id'
# ].tolist()
# patterns = [f'*{sid}[_.]**' for sid in ids]
# load_dataset('MahmoodLab/hest', cache_dir='./hest1k_datasets/mouse_brain', patterns=patterns)

## What you should see now

Inside `./hest1k_datasets/PRAD/` you should find:

```
PRAD/
├── st/                # per-slide AnnData files: MEND139.h5ad, MEND140.h5ad, ...
├── wsis/              # per-slide H&E images:    MEND139.tif,  MEND140.tif,  ...
└── cellvit_seg/       # (optional) cell segmentation outputs
```

Next: `02_build_bank.ipynb` extracts per-spot foundation-model embeddings and selects
the anchor gene panel that GeneRAG queries against.